In [1]:
# DOCUMENT DETAILS

# PURPOSE: GENERAL ARCHITECTURAL DESIGN RENDERS USING GOOGLE GEMINI API
# DATE: 20 FEB 2026
# AUTHOR: FH

In [2]:
# LIBRARIES AND API

In [3]:
# libraries

!pip install -q -U google-genai

# api key

from google.colab import userdata

GEMINI_API_KEY = userdata.get('gemini-render-api-key')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 22.5 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# MODEL PARAMETERS

In [6]:
# system instructions doc

SYSTEM_INSTRUCTIONS_PATH = "/content/drive/MyDrive/alex-llm-docs/system-instructions/sys-inst_img-rend_gen_v3.md"

with open(SYSTEM_INSTRUCTIONS_PATH, "r", encoding="utf-8") as f:
    system_instruction = f.read()


In [7]:
# USER INPUTS - TEXT PROMPT

In [8]:
# text prompt set

prompt = "generate a professional architectural render using the rendered design and ensure it is placed in the setting/context provided"

In [9]:
# USER INPUTS - IMAGES

In [10]:
# image file set

IMG_PATH_A = "/content/drive/MyDrive/alex-llm-docs/sample-render-materials/KITCH RUM DAY.jpeg"  # paste path or leave blank
IMG_PATH_B = "/content/drive/MyDrive/alex-llm-docs/context-images/pylon-context-day-v2.1.jpg"
IMG_PATH_C = ""

# image description set

from PIL import Image
import os

contents = []

if IMG_PATH_A and os.path.exists(IMG_PATH_A):
    img_a = Image.open(IMG_PATH_A)
    contents.append("simple vray render showing color, massing, shade/shadow of a design. this will be placed into the context/setting provided in image B")
    contents.append(img_a)

if IMG_PATH_B and os.path.exists(IMG_PATH_B):
    img_b = Image.open(IMG_PATH_B)
    contents.append("design context/setting. do not change this. add the rendered design from image A to the site context.")
    contents.append(img_b)

if IMG_PATH_C and os.path.exists(IMG_PATH_C):
    img_c = Image.open(IMG_PATH_C)
    contents.append("IMAGE DESCRIPTION")
    contents.append(img_c)

contents.append(prompt)

In [11]:
# RUN

In [ ]:
# model run

from google import genai
from google.genai import types
from PIL import Image
from google.colab import files

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-3-pro-image-preview", # "gemini-3-flash-preview", "gemini-2.5-flash-image"
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="1K"
        )
    ),
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("generated_image.png")
        files.download("generated_image.png")